# Issue #279 — action-aware features and 2023 attribution

Exploratory development analysis only. The independent unit is one Beijing issue; action rows and seeds are nested. All 2024+ outcomes remain sealed.

In [1]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
ROOT=Path.cwd()
while not (ROOT/'results/issue-279').exists() and ROOT != ROOT.parent: ROOT=ROOT.parent
R=ROOT/'results/issue-279'; F=ROOT/'docs/research/figures'; F.mkdir(parents=True,exist_ok=True)
d=json.loads((R/'decision.json').read_text()); features=pd.read_parquet(R/'action_features.parquet')
dictionary=pd.read_csv(R/'feature_dictionary.csv'); profile=pd.read_csv(R/'conditional_oracle_profile.csv')
budget=pd.read_parquet(R/'budget_matched_targets.parquet'); budget_profile=pd.read_csv(R/'conditional_budget_profile.csv')
ledger=pd.read_csv(R/'issue_2023_attribution.csv',dtype={'issue':str}); topb=pd.read_csv(R/'issue_2023_topb_actions.csv',dtype={'issue':str})
trace_summary=pd.read_csv(R/'issue_2023_trace_summary.csv',dtype={'issue':str})
print(json.dumps(d,indent=2)); display(dictionary)

{
  "contract": "BJ-UNIFIED-ACTION-FEATURES-001/v1",
  "development_years": [
    2014,
    2023
  ],
  "development_issues": 620,
  "action_feature_rows": 30368,
  "independent_unit": "Beijing_issue",
  "2023_issues": 53,
  "2023_dynamic_pnl": -1161.343400045458,
  "2023_fixed_pnl": -848.0,
  "2023_lift_pnl": -313.3434000454581,
  "2023_mean_oracle_regret_pnl": 3393.5804847807044,
  "budget_match_reference_stake_cny": 16.0,
  "budget_matched_target_rows": 30368,
  "2023_preoutcome_cell_rows": 43599,
  "2023_preoutcome_ticket_rows": 3264,
  "2023_preoutcome_leg_rows": 20208,
  "2023_topb_action_rows": 265,
  "oracle_parameter_label_valid": false,
  "oracle_bias_warning": "RAW_ARGMAX_CONFOUNDED_BY_K_EXPOSURE_AND_MULTIPLE_ACTION_HINDSIGHT",
  "budget_matched_targets_valid_for_learning": true,
  "marginal_k_diagnostic": "OBSERVED_NOISY_NON_NESTED_ANCHORS",
  "2023_selection_outcome_attribution": "STRUCTURAL_TRACE_ONLY_NONADDITIVE_CAUSAL_EFFECT_NOT_IDENTIFIED",
  "seal": {
    "rows_opened

,feature,stage,formula,earliest_availability,leakage_class
0,stage1_candidate_density,selection,accepted_fraction,decision_epoch,SAFE
1,stage2_probability_concentration,outcome,1-entropy/log(3),decision_epoch,SAFE
2,stage2_divergence_confidence,outcome,mean_abs_divergence/(value_interval+eps),decision_epoch,SAFE
3,stage3_n_fraction,ticket,N/M,decision_epoch,SAFE
4,stage3_log2_k,ticket,log2(K),action,SAFE
5,stage3_requested_turnover,ticket,2*K CNY,action,SAFE
6,stage4_requested_capital_fraction,capital,2*K/prior_capital,decision_epoch,SAFE
7,stage3_effective_coefficient_per_leg,ticket,"0.65^(1/N), comparison only",action,SAFE
8,stage3_log_match_combinations,ticket,"log choose(M,N)",decision_epoch,SAFE
9,stage3_repeat_pressure_proxy,ticket,K/exp(N*mean_entropy),decision_epoch,SAFE_PROXY


## Conditional—not annual—parameter profile

Oracle rows are outcome-aware diagnostics. They describe where the historical return surface lies, not an executable policy. The raw oracle is shown only to demonstrate why its K512 concentration is invalid as a label.

In [2]:
pivot=profile.pivot(index='breadth_bin',columns='divergence_bin',values='oracle_n_median')
display(profile); fig,ax=plt.subplots(figsize=(10,5)); im=ax.imshow(pivot,aspect='auto',cmap='viridis')
ax.set_xticks(range(len(pivot.columns)),pivot.columns,rotation=35,ha='right'); ax.set_yticks(range(len(pivot.index)),pivot.index)
ax.set_xlabel('Mean absolute divergence quartile'); ax.set_ylabel('Issue breadth quartile'); ax.set_title('Outcome-aware median oracle N | issue Data')
fig.colorbar(im,ax=ax,label='Median oracle N'); plt.tight_layout(); plt.savefig(F/'bj279_conditional_oracle_n.png',dpi=160); plt.show()

,breadth_bin,divergence_bin,issues,oracle_n_median,oracle_k_median,oracle_utility_mean,oracle_positive_fraction
0,"(6.999, 139.0]","(0.030899999999999997, 0.0429]",62,7.0,512.0,260.459622,1.000000
1,"(6.999, 139.0]","(0.0429, 0.0453]",22,7.0,512.0,311.913562,1.000000
2,"(6.999, 139.0]","(0.0453, 0.0478]",26,7.0,512.0,222.900158,1.000000
3,"(6.999, 139.0]","(0.0478, 0.0677]",46,7.0,512.0,397.357296,1.000000
4,"(139.0, 234.5]","(0.030899999999999997, 0.0429]",62,7.0,512.0,251.520324,0.983871
5,"(139.0, 234.5]","(0.0429, 0.0453]",18,7.0,512.0,469.672994,0.944444
6,"(139.0, 234.5]","(0.0453, 0.0478]",28,7.0,512.0,240.866802,0.928571
7,"(139.0, 234.5]","(0.0478, 0.0677]",46,7.0,512.0,220.178899,0.956522
8,"(234.5, 417.25]","(0.030899999999999997, 0.0429]",16,7.0,512.0,298.252907,1.000000
9,"(234.5, 417.25]","(0.0429, 0.0453]",52,7.0,512.0,296.357287,0.980769


## Budget-matched action targets and marginal K

Every active action is rescaled to the same CNY16 reference stake before comparison. This removes the mechanical advantage of larger K while preserving the return-per-yuan label. Marginal K observations are noisy because adjacent ticket samples are not nested; they are diagnostics for a later expected-curve model, not a stopping policy by themselves.

In [3]:
n7=(budget[budget.n.eq(7)&budget.generator.ne('NO_BET')]
    .groupby(['generator','k']).agg(mean_budget_pnl=('budget_matched_pnl','mean'),median_budget_pnl=('budget_matched_pnl','median'),positive_fraction=('budget_matched_pnl',lambda x:(x>0).mean())).reset_index())
display(n7)
fig,ax=plt.subplots(figsize=(10,5))
for generator,g in n7.groupby('generator'):
    ax.plot(g.k,g.mean_budget_pnl,marker='o',label=generator)
ax.axhline(0,color='black',lw=.8); ax.set_xscale('log',base=2); ax.set_xticks(sorted(n7.k.unique()),sorted(n7.k.unique()))
ax.set_xlabel('K anchor'); ax.set_ylabel('Mean PnL at matched CNY16 budget'); ax.set_title('Development N7 budget-matched K curves'); ax.legend()
plt.tight_layout(); plt.savefig(F/'bj279_budget_matched_n7_k.png',dpi=160); plt.show()

,generator,k,mean_budget_pnl,median_budget_pnl,positive_fraction
0,TOP_VALUE,8,-9.888586,-16.000000,0.003226
1,TOP_VALUE,32,-9.888586,-16.000000,0.003226
2,TOP_VALUE,128,-9.888586,-16.000000,0.003226
3,TOP_VALUE,512,-9.888586,-16.000000,0.003226
4,WEIGHTED_WOR,8,-3.948107,-16.000000,0.019355
5,WEIGHTED_WOR,32,2.185404,-16.000000,0.077419
6,WEIGHTED_WOR,128,-2.034161,-16.000000,0.229032
7,WEIGHTED_WOR,512,-2.194381,-8.470751,0.314516
8,WEIGHTED_WR,8,2.103747,-16.000000,0.019355
9,WEIGHTED_WR,32,-1.253785,-16.000000,0.064516


## 2023 per-issue replay and regret

Decisions are frozen pre-outcome in #209. Realized results and oracle actions are joined only for development diagnosis.

In [4]:
display(ledger[['issue','action_key','mean_pnl','fixed_pnl','lift_pnl_vs_fixed','oracle_regret_pnl','failure_stage']].head(20))
fig,ax=plt.subplots(figsize=(11,5)); ax.plot(ledger.dynamic_capital,label='dynamic'); ax.plot(ledger.fixed_capital,label='fixed N7/K8')
ax.axhline(100000,color='black',ls='--',lw=.8); ax.set_xlabel('2023 issue sequence'); ax.set_ylabel('Pre-tax capital CNY'); ax.legend(); ax.set_title('2023 development replay')
plt.tight_layout(); plt.savefig(F/'bj279_2023_capital_attribution.png',dpi=160); plt.show()

,issue,action_key,mean_pnl,fixed_pnl,lift_pnl_vs_fixed,oracle_regret_pnl,failure_stage
0,22112,N3_K8_WEIGHTED_WOR,8.132566,-16.0,24.132566,2679.935728,NO_FAILURE_VS_FIXED
1,22114,N5_K8_WEIGHTED_WOR,-16.000000,-16.0,0.000000,71.510525,NO_FAILURE_VS_FIXED
2,22123,N5_K8_WEIGHTED_WR,-16.000000,-16.0,0.000000,1485.776749,NO_FAILURE_VS_FIXED
3,22124,N7_K8_WEIGHTED_WR,-16.000000,-16.0,0.000000,1514.199602,NO_FAILURE_VS_FIXED
4,22126,N7_K8_WEIGHTED_WR,-16.000000,-16.0,0.000000,4557.413671,NO_FAILURE_VS_FIXED
5,23011,N3_K8_WEIGHTED_WOR,35.856648,-16.0,51.856648,1667.677234,NO_FAILURE_VS_FIXED
6,23012,N7_K8_WEIGHTED_WR,-16.000000,-16.0,0.000000,2495.065308,NO_FAILURE_VS_FIXED
7,23013,N5_K8_WEIGHTED_WR,-16.000000,-16.0,0.000000,1277.687757,NO_FAILURE_VS_FIXED
8,23014,N5_K8_WEIGHTED_WOR,-16.000000,-16.0,0.000000,1197.122620,NO_FAILURE_VS_FIXED
9,23021,N7_K8_WEIGHTED_WR,-16.000000,-16.0,0.000000,3384.421697,NO_FAILURE_VS_FIXED


In [5]:
stages=ledger.failure_stage.value_counts(); display(stages.rename('issues').to_frame())
ax=stages.sort_values().plot.barh(figsize=(9,4),color='#6b8e23'); ax.set_title('2023 realized failure attribution versus fixed control'); ax.set_xlabel('Issues')
plt.tight_layout(); plt.savefig(F/'bj279_2023_failure_stages.png',dpi=160); plt.show()
print('Selection attribution:',ledger.selection_attribution.unique().tolist()); print('Outcome attribution:',ledger.outcome_attribution.unique().tolist())

,issues
failure_stage,
NO_FAILURE_VS_FIXED,51
JOINT_N_K_GENERATOR,2


Selection attribution: ['STRUCTURAL_MATCH_SET_CHANGED_NONADDITIVE']
Outcome attribution: ['STRUCTURAL_OUTCOME_MIX_CHANGED_NONADDITIVE']


## Pre-outcome structural attribution

The replay now persists all 2023 cell values, Top-B action alternatives, complete ticket identities and every ticket leg before outcomes are joined. Match, outcome and ticket Jaccard scores show where the learned action structurally diverges from fixed N7/K8 TOP_VALUE. These scores do not pretend that nonlinear N-fold PnL can be additively assigned to individual stages.

In [6]:
display(ledger[['issue','action_key','lift_pnl_vs_fixed','match_set_jaccard','leg_identity_jaccard','ticket_identity_jaccard','shared_match_modal_outcome_disagreement']].head(20))
fig,ax=plt.subplots(figsize=(9,5)); ax.scatter(ledger.match_set_jaccard,ledger.lift_pnl_vs_fixed,alpha=.75)
ax.axhline(0,color='black',lw=.8); ax.set_xlabel('Chosen vs fixed match-set Jaccard'); ax.set_ylabel('Realized matched PnL lift (CNY)'); ax.set_title('2023 structural divergence and realized lift')
plt.tight_layout(); plt.savefig(F/'bj279_2023_structure_vs_lift.png',dpi=160); plt.show()
print('Pre-outcome artifacts:', {'Top-B rows':len(topb),'issues':trace_summary.issue.nunique(),'cell rows':d['2023_preoutcome_cell_rows'],'ticket rows':d['2023_preoutcome_ticket_rows'],'leg rows':d['2023_preoutcome_leg_rows']})

,issue,action_key,lift_pnl_vs_fixed,match_set_jaccard,leg_identity_jaccard,ticket_identity_jaccard,shared_match_modal_outcome_disagreement
0,22112,N3_K8_WEIGHTED_WOR,24.132566,0.014085,0.000000,0.0,1.000000
1,22114,N5_K8_WEIGHTED_WOR,0.000000,0.064935,0.038835,0.0,0.200000
2,22123,N5_K8_WEIGHTED_WR,0.000000,0.098361,0.041667,0.0,0.333333
3,22124,N7_K8_WEIGHTED_WR,0.000000,0.066667,0.035971,0.0,0.428571
4,22126,N7_K8_WEIGHTED_WR,0.000000,0.084507,0.023810,0.0,0.666667
5,23011,N3_K8_WEIGHTED_WOR,51.856648,0.068966,0.042254,0.0,0.500000
6,23012,N7_K8_WEIGHTED_WR,0.000000,0.026786,0.013245,0.0,0.666667
7,23013,N5_K8_WEIGHTED_WR,0.000000,0.019048,0.008264,0.0,1.000000
8,23014,N5_K8_WEIGHTED_WOR,0.000000,0.088235,0.039216,0.0,0.333333
9,23021,N7_K8_WEIGHTED_WR,0.000000,0.022556,0.012658,0.0,0.333333


Pre-outcome artifacts: {'Top-B rows': 265, 'issues': 53, 'cell rows': 43599, 'ticket rows': 3264, 'leg rows': 20208}


## Interpretation

The first feature slice is action-aware and leakage-safe. Budget matching removes the raw-oracle K exposure artifact, but the resulting targets remain extremely skewed: most action/issue outcomes lose the full CNY16 reference budget and a few large hits dominate means. Therefore the next model must learn conditional return distributions and downside probabilities, not regress a single mean or imitate a hindsight argmax.

The new trace makes match-selection and outcome-direction changes observable, while their separate causal PnL contributions remain non-identifiable because a ticket pays only when all N legs hit. 2023 remains development evidence and grants no promotion authority.